# Lektion 03 - Agentiske Designmønstre

I denne lektion udforsker vi tre grundlæggende designmønstre til at opbygge effektive AI-agenter:

1. **Klare Agentinstruktioner** — Udformning af præcise, rolle-definerende prompt, der styrer agentens adfærd
2. **Struktureret Output med Pydantic-modeller** — Sikring af, at agenter returnerer forudsigelig, valideret data
3. **Enkelt Ansvarsagent** — Design af fokuserede agenter, der hver især gør én ting godt

Vi vil anvende hvert mønster på et **rejsemålsanbefalingsscenarie**, hvor vi gradvist bygger et system, der kan foreslå destinationer, tjekke tilgængelighed og håndtere logistik.


## Opsætning


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mønster 1: Klare agentinstruktioner

Det mest effektive mønster er også det enkleste: at skrive klare, detaljerede instruktioner til din agent.

Gode instruktioner definerer:
- **Hvem** agenten er (persona og tone)
- **Hvad** den skal gøre (trin-for-trin ansvarsområder)
- **Hvordan** den skal opføre sig (begrænsninger og stil)

Nedenfor opretter vi en rejsekonsulentagent med eksplicitte instruktioner, der former hvert svar, den producerer.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Mønster 2: Struktureret output med Pydantic-modeller

Frit tekstformat er nyttigt til samtale, men efterfølgende systemer har brug for strukturerede data.
Ved at kombinere **Pydantic-modeller** med en **værktøjsfunktion** kan vi:

- Definere et præcist skema for agentens output
- Validere svar automatisk
- Integrere agentresultater pålideligt i applikationslogik

Nøglen til håndhævelse er at videregive `response_format`, når vi kører agenten. Dette tvinger
modellen til at returnere et valideret `TravelRecommendations`-objekt (tilgængeligt på `response.value`)
i stedet for frit tekstformat. `get_destination_details`-værktøjet returnerer også en typet
`DestinationRecommendation`, så dataene forbliver strukturerede hele vejen igennem.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Mønster 3: Enkeltansvarsagenter

Komplekse opgaver drager fordel af at opdele arbejdet på tværs af flere fokuserede agenter, hver med et enkelt ansvar:

- En **Destinationsekspert**, der kender til steder og tilgængelighed
- En **Logistikplanlægger**, der håndterer fly, hoteller og rejseplaner

Dette spejler softwareingeniørprincippet om *adskillelse af bekymringer* — hver agent er nemmere at teste, vedligeholde og forbedre uafhængigt.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Resumé

I denne lektion anvendte vi tre agentorienterede designmønstre på et rejseanbefalingsscenarie:

| Mønster | Nøgleidé | Fordel |
|---|---|---|
| **Klare instruktioner** | Definér persona, ansvar og begrænsninger på forhånd | Ensartet, brand-kompatibel agentadfærd |
| **Struktureret output** | Brug Pydantic-modeller som svarformat | Validerede, maskinlæselige resultater |
| **Single ansvar** | Giv hver agent én fokuseret opgave | Nemmere at teste, vedligeholde og sammensætte |

Disse mønstre kan kombineres naturligt — du kan kombinere klare instruktioner med struktureret output inden for en single-ansvarsagent for at opbygge robuste, produktionsklare systemer.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
